## Dicionário de Dados

Abaixo estão as **principais colunas** do dataset de gastos públicos federais, com seus tipos e descrições:

| Coluna | Type | Comment |
|--------|------|--------|
| ID_ANO | INTEGER | Ano de referência da despesa (2024 ou 2025) |
| ID_MES | INTEGER | Mês de referência (1 a 12) |
| data_referencia | DATE | Data criada no formato YYYY-MM-01 para análises temporais |
| ORGAO_CODIGO | INTEGER | Código numérico do órgão federal |
| ORGAO_DESCRICAO | STRING | Nome completo do órgão (ex: MINISTERIO DA FAZENDA) |
| ORGAO_CNPJ | STRING | CNPJ do órgão federal |
| Poder_Orgao | STRING | Poder ao qual o órgão pertence (EXE, LEG, JUD) |
| UNIDADE_GESTORA_CODIGO | INTEGER | Código da unidade gestora responsável |
| UNIDADE_GESTORA_DESCRICAO | STRING | Nome da unidade gestora |
| FUNCAO | STRING | Função de governo (ex: SAUDE, EDUCACAO, PREVIDENCIA SOCIAL) |
| SUBFUNCAO | STRING | Subfunção detalhada da despesa |
| PROGRAMA_PT | STRING | Código do programa de trabalho |
| NO_PROGRAMA_PT | STRING | Nome do programa (ex: BOLSA FAMILIA, OPERACOES ESPECIAIS) |
| ACAO_PT | STRING | Código da ação programática |
| NO_ACAO_PT | STRING | Nome da ação orçamentária |
| DOTACAO_INICIAL | DECIMAL(18,2) | Valor inicial previsto no orçamento (em Reais) |
| DOTACAO_ATUALIZADA | DECIMAL(18,2) | Valor atualizado após remanejamentos (em Reais) |
| DESPESAS_EMPENHADAS | DECIMAL(18,2) | Total de despesas empenhadas no período (em Reais) |
| DESPESAS_LIQUIDADAS | DECIMAL(18,2) | Total de despesas liquidadas no período (em Reais) |
| DESPESAS_PAGAS | DECIMAL(18,2) | Total de despesas pagas no período (em Reais) |
| PAGAMENTOS_TOTAIS | DECIMAL(18,2) | **Coluna principal:** soma de todos os pagamentos executados (em Reais) |

**Total de colunas:** 39 (incluindo metadados adicionais de classificação orçamentária)

**Notas:**
- Valores financeiros estão em Reais (BRL)
- Valores negativos são legítimos (representam anulações ou devoluções)
- Coluna `PAGAMENTOS_TOTAIS` é a métrica principal para análises de execução

## Linhagem de Dados

Este projeto segue a **arquitetura Medallion**, organizando os dados em camadas progressivas de qualidade e agregação.

---

### 1. Fontes (Staging)

**Nome dos arquivos:** 
- `gastos2024.csv` (150.423 registros)
- `gastos2025.csv` (147.075 registros)

**Origem:** Portal da Transparência - Governo Federal  
**URL:** https://portaldatransparencia.gov.br/download-de-dados/despesas-execucao

**Formato técnico:**
- Separador: `;` (ponto e vírgula)
- Encoding: ISO-8859-1 (Latin-1)
- Header: Sim (primeira linha)
- Valores financeiros: formato brasileiro com vírgula decimal

**Volume criado:** `MVP_gastos_publicos.staging.dados_gastos_publicos` (Unity Catalog Volume)

---

### 2. Schema Bronze

**Tabela:** `MVP_gastos_publicos.bronze.gastos_publicos`

**Transformações aplicadas:**
- Leitura dos 2 arquivos CSV com encoding ISO-8859-1
- União (UNION) dos datasets 2024 + 2025
- Todas as colunas mantidas como **STRING** (dados brutos)
- Preservação de valores originais (sem limpeza)

**Registros:** 297.498

**Objetivo:** Camada de dados brutos, imutável, para auditoria e reprocessamento

---

### 3. Schema Silver

**Tabela:** `MVP_gastos_publicos.silver.gastos_publicos`

**Transformações aplicadas:**
- **Conversão de tipos:** 
  - 8 colunas financeiras: STRING → DECIMAL(18,2)
  - 10 colunas de códigos: STRING → INTEGER
  - Criação de `data_referencia`: DATE (formato YYYY-MM-01)
- **Limpeza de dados:** 
  - Remoção de valores malformados ('N/A', '00QD', strings inválidas)
  - Conversão de formato brasileiro (vírgula → ponto decimal)
  - Substituição de valores inválidos por NULL
- **Remoção de duplicados:** Eliminados 27.000 registros duplicados

**Registros:** 270.498 (após limpeza)

**Objetivo:** Camada de dados limpos e validados, pronta para análise

---

### 4. Armazenamento Gold

**Tabelas criadas:**

#### 4.1 `gastos_publicos` (tabela base)
- Dados da camada Silver filtrados e otimizados
- Registros: 270.498

#### 4.2 `gastos_por_orgao_ano` (agregação por órgão)
```sql
GROUP BY ORGAO_DESCRICAO, Poder_Orgao, ID_ANO
SUM(PAGAMENTOS_TOTAIS)
```
- Registros: ~1.200 (órgãos × anos)

#### 4.3 `gastos_por_funcao` (agregação por função de governo)
```sql
GROUP BY FUNCAO, ID_ANO
SUM(PAGAMENTOS_TOTAIS)
```
- Registros: ~60 (funções × anos)

#### 4.4 `evolucao_mensal` (série temporal)
```sql
GROUP BY ID_ANO, ID_MES, data_referencia
SUM(PAGAMENTOS_TOTAIS)
```
- Registros: 24 (12 meses × 2 anos)

**Objetivo:** Tabelas agregadas otimizadas para responder as 5 perguntas de negócio com performance

In [0]:
%sql
-- Listar todos os schemas do catálogo
SHOW SCHEMAS IN MVP_gastos_publicos;

In [0]:
%sql
-- Tabelas na camada Bronze
USE CATALOG MVP_gastos_publicos;
SHOW TABLES IN bronze;

In [0]:
%sql
-- Tabelas na camada Silver
SHOW TABLES IN silver;

In [0]:
%sql
-- Tabelas na camada Gold
SHOW TABLES IN gold;

In [0]:
%sql
-- Descrever estrutura completa da tabela principal (Silver)
DESCRIBE TABLE EXTENDED silver.gastos_publicos;

In [0]:
%sql
-- Visualizar amostra de dados da camada Silver
SELECT 
    ID_ANO,
    ID_MES,
    data_referencia,
    ORGAO_DESCRICAO,
    Poder_Orgao,
    FUNCAO,
    NO_PROGRAMA_PT,
    ROUND(PAGAMENTOS_TOTAIS, 2) as PAGAMENTOS_TOTAIS
FROM silver.gastos_publicos
WHERE PAGAMENTOS_TOTAIS IS NOT NULL
ORDER BY PAGAMENTOS_TOTAIS DESC
LIMIT 10;

## Resumo da Arquitetura

### Pipeline Medallion Implementado

```
📁 STAGING (Unity Catalog Volume)
   ↓
🥉 BRONZE (Dados Brutos)
   ├─ gastos_publicos (297k registros, 39 colunas STRING)
   ↓
🥈 SILVER (Dados Limpos)
   ├─ gastos_publicos (270k registros, tipos validados)
   ↓
🥇 GOLD (Dados Agregados)
   ├─ gastos_publicos (base filtrada)
   ├─ gastos_por_orgao_ano (por órgão + ano)
   ├─ gastos_por_funcao (por função)
   └─ evolucao_mensal (série temporal)
```

---

### Métricas do Projeto

| Métrica | Valor |
|---------|-------|
| **Registros brutos** | 297.498 |
| **Registros após limpeza** | 270.498 |
| **Colunas totais** | 39 |
| **Período coberto** | 24 meses (2024-2025) |
| **Órgãos analisados** | ~600 órgãos federais |
| **Funções de governo** | 28 funções principais |
| **Volume total de gastos** | R$ 9,88 trilhões |

---

### Qualidade dos Dados

✅ **Validações aplicadas:**
- Verificação de valores nulos em colunas críticas
- Detecção e tratamento de valores negativos legítimos
- Identificação de duplicados (4.402 registros = 1,6%)
- Completude temporal confirmada (24 meses sem gaps)
- Valores extremos documentados e validados

✅ **Governança:**
- Catálogo Unity Catalog com controle de acesso
- Schemas separados por camada de qualidade
- Tabelas Delta Lake com histórico de versões
- Documentação completa em notebooks

---

### Próximos Passos

Este catálogo serve como base para:
1. ✅ Análise das 5 perguntas de negócio (notebook **7 - Analise**)
2. 🔄 Atualizações incrementais com novos períodos
3. 📊 Dashboards executivos de acompanhamento
4. 🤖 Modelos preditivos de execução orçamentária

---

  
**Responsável:** Christopher Silva  
**Projeto:** MVP PUC-RJ - Análise de Gastos Públicos Federais